# 10 — Asian Options

Arithmetic and geometric averaging, multiple engines:
- **Geometric analytic** (closed-form)
- **Turnbull-Wakeman** (arithmetic, moment-matching)
- **Choi** (arithmetic, improved analytic)
- **Lévy** (continuous arithmetic)
- **Monte Carlo** (arithmetic, with control variate)

Plus AD Greeks and `vmap` batch pricing.

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.asian import geometric_asian_price, turnbull_wakeman_price
from ql_jax.engines.analytic.asian_choi import choi_asian_price, levy_asian_price
from ql_jax.engines.mc.asian import mc_asian_arithmetic_bs

S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.02, 0.30
N_FIX = 12   # monthly fixings

---
## 1. QuantLib Reference

In [ ]:
today = QL.Date(15, 6, 2024)
QL.Settings.instance().evaluationDate = today
maturity = QL.Date(15, 6, 2025)

dc = QL.Actual365Fixed()
cal = QL.NullCalendar()

spot_h = QL.QuoteHandle(QL.SimpleQuote(S))
r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, r, dc))
q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, q, dc))
vol_ts = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(today, cal, sigma, dc))
bsm    = QL.BlackScholesMertonProcess(spot_h, q_ts, r_ts, vol_ts)

# Monthly fixing dates
import datetime
fixing_dates = []
for i in range(1, N_FIX + 1):
    d = today + QL.Period(i, QL.Months)
    fixing_dates.append(d)

# Geometric discrete-average Asian
payoff_call = QL.PlainVanillaPayoff(QL.Option.Call, K)
exercise = QL.EuropeanExercise(maturity)
asian_geo = QL.DiscreteAveragingAsianOption(
    QL.Average.Geometric, 0.0, 0, fixing_dates, payoff_call, exercise)
asian_geo.setPricingEngine(QL.AnalyticDiscreteGeometricAveragePriceAsianEngine(bsm))
ql_geo = asian_geo.NPV()

# Arithmetic discrete-average (MC)
asian_arith = QL.DiscreteAveragingAsianOption(
    QL.Average.Arithmetic, 0.0, 0, fixing_dates, payoff_call, exercise)
asian_arith.setPricingEngine(QL.MCDiscreteArithmeticAPEngine(bsm, 'pseudorandom',
    requiredSamples=100000))
ql_mc = asian_arith.NPV()

print(f"QL geometric Asian call : {ql_geo:.6f}")
print(f"QL MC arithmetic Asian  : {ql_mc:.6f}")

---
## 2. ql-jax Analytic Engines

In [ ]:
jax_geo = float(geometric_asian_price(S, K, T, r, q, sigma, N_FIX, option_type='call'))
jax_tw  = float(turnbull_wakeman_price(S, K, T, r, q, sigma, N_FIX, option_type='call'))
jax_choi = float(choi_asian_price(S, K, T, r, q, sigma, N_FIX, option_type='call'))
jax_levy = float(levy_asian_price(S, K, T, r, q, sigma, option_type='call'))

print(f"Geometric (ql-jax)      : {jax_geo:.6f}")
print(f"Turnbull-Wakeman        : {jax_tw:.6f}")
print(f"Choi                    : {jax_choi:.6f}")
print(f"Lévy (continuous)       : {jax_levy:.6f}")

---
## 3. Monte Carlo

In [ ]:
jax_mc, jax_se = mc_asian_arithmetic_bs(
    S, K, T, r, q, sigma, option_type=1,
    n_fixings=N_FIX, n_paths=200_000, antithetic=True, control_variate=True)
jax_mc, jax_se = float(jax_mc), float(jax_se)

print(f"MC arithmetic Asian     : {jax_mc:.6f}  ± {jax_se:.6f}")

---
## 4. Comparison Table

In [ ]:
rows = [
    ("Geometric (QL)", ql_geo, jax_geo, ""),
    ("Turnbull-Wakeman", "-", jax_tw, ""),
    ("Choi", "-", jax_choi, ""),
    ("Lévy (continuous)", "-", jax_levy, ""),
    ("MC arithmetic (QL)", ql_mc, jax_mc, ""),
]

print(f"\n{'Engine':<25s} {'QL':>12s} {'ql-jax':>12s} {'Diff':>12s}")
print("-" * 65)
for label, ql_v, jax_v, _ in rows:
    if isinstance(ql_v, str):
        print(f"{label:<25s} {'n/a':>12s} {jax_v:>12.6f}")
    else:
        print(f"{label:<25s} {ql_v:>12.6f} {jax_v:>12.6f} {abs(ql_v - jax_v):>12.6f}")

---
## 5. Put Options

In [ ]:
geo_put = float(geometric_asian_price(S, K, T, r, q, sigma, N_FIX, option_type='put'))
tw_put  = float(turnbull_wakeman_price(S, K, T, r, q, sigma, N_FIX, option_type='put'))

print(f"Geometric put : {geo_put:.6f}")
print(f"TW put        : {tw_put:.6f}")

---
## 6. AD Greeks

In [ ]:
import matplotlib.pyplot as plt

def asian_call(spot, vol):
    return turnbull_wakeman_price(spot, K, T, r, q, vol, N_FIX, option_type='call')

delta_fn = jax.grad(asian_call, argnums=0)
vega_fn  = jax.grad(asian_call, argnums=1)

spots = jnp.linspace(70, 130, 60)
deltas = [float(delta_fn(s, sigma)) for s in spots]
vegas  = [float(vega_fn(S, v)) for v in jnp.linspace(0.10, 0.60, 60)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(np.array(spots), deltas, 'b-', linewidth=2)
ax1.set_xlabel('Spot'); ax1.set_ylabel('Delta')
ax1.set_title('Asian Call Delta (Turnbull-Wakeman)')
ax1.grid(True, alpha=0.3)

ax2.plot(np.linspace(10, 60, 60), vegas, 'r-', linewidth=2)
ax2.set_xlabel('Vol (%)')
ax2.set_ylabel('Vega')
ax2.set_title('Asian Call Vega')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 7. Convergence: Fixings Count

In [ ]:
fixings = [4, 12, 26, 52, 100, 252]
geo_prices = [float(geometric_asian_price(S, K, T, r, q, sigma, n)) for n in fixings]
tw_prices  = [float(turnbull_wakeman_price(S, K, T, r, q, sigma, n)) for n in fixings]

plt.figure(figsize=(8, 5))
plt.plot(fixings, geo_prices, 'bo-', label='Geometric')
plt.plot(fixings, tw_prices, 'rs-', label='Turnbull-Wakeman')
plt.axhline(y=float(levy_asian_price(S, K, T, r, q, sigma, 'call')),
            color='gray', linestyle='--', label='Continuous (Lévy)')
plt.xlabel('Number of Fixings')
plt.ylabel('Price')
plt.title('Asian Option Price vs Fixing Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 8. vmap: Batch Pricing

In [ ]:
batch_tw = jax.jit(jax.vmap(lambda s: turnbull_wakeman_price(s, K, T, r, q, sigma, N_FIX, 'call')))

spots_batch = jnp.linspace(80.0, 120.0, 1000)
_ = batch_tw(spots_batch)  # warmup

tw_t, prices = timed_ms(batch_tw, spots_batch)
print(f"1,000 TW prices in {tw_t:.1f} ms")
print(f"Range: {float(prices.min()):.4f} — {float(prices.max()):.4f}")

---
## 9. Exercises

1. **Put-call relationship**: For geometric Asians, verify the put-call parity relationship.
2. **MC convergence**: Plot MC price ± 2 stderr vs number of paths; overlay the analytic TW price.
3. **Gamma surface**: Compute $\partial^2 V / \partial S^2$ across a spot × vol grid using `jax.hessian`.

---
*End of Notebook 10*